<a href="https://colab.research.google.com/github/praveena-muvva/hf-llm/blob/main/05_finetuning_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name()}")

In [ ]:
#Load model and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token #GPT-2 needs this

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map = "auto"
)
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
#Applying LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8, #rank - size of adapter matrices
    lora_alpha=16, #scaling factor
    target_modules=["c_attn"], #Which layer to adapt - GPT specific
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

#Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#Preparing data
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:1000]")

def tokenize_function(examples):
  return tokenizer(
    examples["text"],
    truncation=True,
    max_length=512,
    padding="max_length"
  )


tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

In [ ]:
#Train with LoRA
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir = "./gpt2-lora",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=100
)

#Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

#Train
trainer.train()

In [ ]:
#Testing the model
prompt = "This movie was"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=1,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Save only the LoRA adapters (tiny file!)
model.save_pretrained("./my-lora-adapters")

# Later, you can load them:
# from peft import PeftModel
# base_model = AutoModelForCausalLM.from_pretrained("gpt2")
# model = PeftModel.from_pretrained(base_model, "./my-lora-adapters")